# Phase 3.3 – Modèle de Réponse aux Promotions

## Objectif
Prédire l'impact d'une promotion sur les ventes afin d'optimiser les décisions promotionnelles et maximiser le ROI marketing sous contrainte de budget réduit.

## Problème
**Régression** : Prédire le volume de ventes (`NB_SALES`) lors d'un mois promotionnel en fonction des caractéristiques de la promotion et du contexte.

## Source de données
- `ANYCOMPANY_LAB.ANALYTICS.MARKETING_PERFORMANCE_MART`

## Algorithmes testés
- Random Forest Regressor
- Gradient Boosting Regressor
- Linear Regression (baseline)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Librairies chargées avec succès ✅')

## 1. Chargement des données

In [ ]:
# Charger le fichier CSV exporté depuis Snowflake :
# ANYCOMPANY_LAB.ANALYTICS.MARKETING_PERFORMANCE_MART
df = pd.read_csv('marketing_performance_mart.csv')

df.columns = df.columns.str.upper()
df['MONTH_DATE'] = pd.to_datetime(df['MONTH_DATE'])
df['MONTH'] = df['MONTH_DATE'].dt.month
df['YEAR']  = df['MONTH_DATE'].dt.year
df = df[df['NB_SALES'] > 0].reset_index(drop=True)

print(f'Données chargées : {df.shape[0]} observations, {df.shape[1]} colonnes')
df.head(10)

In [ ]:
df.describe()

## 2. Feature Engineering

In [ ]:
# Encodage de la région
le = LabelEncoder()
df['REGION_ENCODED'] = le.fit_transform(df['REGION'])

# Feature : présence d'une promotion (binaire)
df['HAS_PROMOTION'] = (df['NB_PROMOTIONS'] > 0).astype(int)

# Feature : présence d'une campagne (binaire)
df['HAS_CAMPAIGN'] = (df['NB_CAMPAIGNS'] > 0).astype(int)

# Feature : saison
df['SEASON'] = df['MONTH'].apply(lambda m:
    0 if m in [12, 1, 2] else
    1 if m in [3, 4, 5] else
    2 if m in [6, 7, 8] else 3
)

# Sélection des features
feature_cols = [
    'NB_PROMOTIONS',
    'AVG_DISCOUNT_PERCENTAGE',
    'NB_CAMPAIGNS',
    'TOTAL_CAMPAIGN_BUDGET',
    'AVG_CONVERSION_RATE',
    'HAS_PROMOTION',
    'HAS_CAMPAIGN',
    'REGION_ENCODED',
    'MONTH',
    'SEASON'
]

target_col = 'NB_SALES'

X = df[feature_cols].fillna(0)
y = df[target_col]

print(f'Features : {len(feature_cols)}')
print(f'Observations : {len(X)}')
print(f'\nCorrelation avec NB_SALES :')
corr = df[feature_cols + [target_col]].corr()[target_col].sort_values(ascending=False)
print(corr.drop(target_col).to_string())

## 3. Split Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train : {len(X_train)} observations')
print(f'Test  : {len(X_test)} observations')

## 4. Entraînement des modèles

In [ ]:
models = {
    'Linear Regression (baseline)': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    cv   = cross_val_score(model, X, y, cv=5, scoring='r2').mean()

    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_R2': cv}
    print(f'\n{name}')
    print(f'  MAE  : {mae:.4f}')
    print(f'  RMSE : {rmse:.4f}')
    print(f'  R²   : {r2:.4f}')
    print(f'  CV R²: {cv:.4f}')

## 5. Comparaison des modèles

In [ ]:
results_df = pd.DataFrame(results).T

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    colors_bar = ['#2196F3', '#4CAF50', '#FF9800']
    bars = axes[i].bar(results_df.index, results_df[metric], color=colors_bar)
    axes[i].set_title(f'Comparaison – {metric}')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=15)
    axes[i].grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, results_df[metric]):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Comparaison des modèles – Réponse aux Promotions', fontsize=13)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_model_name = results_df['R2'].idxmax()
print(f'\n✅ Meilleur modèle : {best_model_name} (R² = {results_df.loc[best_model_name, "R2"]:.4f})')

## 6. Importance des features

In [ ]:
# Feature importance du Random Forest
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors_imp = ['#e74c3c' if imp > importances.median() else '#3498db' for imp in importances]
importances.plot(kind='barh', color=colors_imp)
plt.title('Importance des features – Random Forest')
plt.xlabel('Importance')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('feature_importance_promo.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 3 features les plus importantes :')
for feat, imp in importances.sort_values(ascending=False).head(3).items():
    print(f'  {feat} : {imp:.4f}')

## 7. Prédictions vs Réalité

In [ ]:
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_best, alpha=0.5, color='#3498db', s=30)
min_val, max_val = min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Prédiction parfaite')
plt.xlabel('Ventes réelles')
plt.ylabel('Ventes prédites')
plt.title(f'Prédictions vs Réalité – {best_model_name}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('predictions_vs_reality.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Interprétation et recommandations

In [ ]:
print('=== SYNTHÈSE DU MODÈLE DE RÉPONSE AUX PROMOTIONS ===')
print(f'\nMeilleur modèle : {best_model_name}')
print(f'R² : {results_df.loc[best_model_name, "R2"]:.4f}')
print(f'MAE : {results_df.loc[best_model_name, "MAE"]:.2f} ventes en moyenne')

print('\n=== RECOMMANDATIONS MARKETING ===')
top_features = importances.sort_values(ascending=False).head(3)
print('\nLes 3 leviers promotionnels les plus influents sur les ventes :')
for i, (feat, imp) in enumerate(top_features.items(), 1):
    print(f'  {i}. {feat} (importance : {imp:.4f})')

print('\nConclusion : Le modèle permet d\'estimer l\'impact d\'une promotion')
print('avant son lancement pour optimiser les ressources marketing.')

In [ ]:
print('✅ Notebook promotion_response_model terminé avec succès.')